# Train Data Analysis — Part 1

## Notebook Overview

In this notebook, we performed the **initial exploratory analysis of the training dataset**.  
The objective of this analysis is to understand the **structure, feature types, and distributions of key variables** before proceeding to modeling.

The following steps were completed:

- Examined dataset structure using **`info()` and null value checks**
- Classified features into **categorical, continuous, count, and time-related groups**
- Explored **demographic variables** such as sex, language, insurance type, and age group
- Analyzed **visit context features** including arrival mode, transport origin, pain location, mental status, and chief complaint system
- Investigated **healthcare utilization variables** such as prior ED visits, hospital admissions, active medications, and comorbidity counts

These analyses provide insight into **patient demographics, clinical presentation patterns, and healthcare usage behavior**, which are important for understanding factors that may influence **triage acuity prediction**.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as pyplot
import seaborn as sns


In [ ]:
df_train = pd.read_csv("../data/train.csv")
df_train.head()

In [ ]:
df_info = pd.DataFrame({
    "Column": df_train.columns,
    "Non-Null Count": df_train.count().values,
    "Null Count": df_train.isnull().sum().values,
    "Data Type": df_train.dtypes.values
})

df_info

In [ ]:
# Identifier columns
id_cols = [
    "patient_id",
    "site_id",
    "triage_nurse_id"
]

# Time-related features
time_cols = [
    "arrival_hour",
    "arrival_day",
    "arrival_month",
    "arrival_season",
    "shift"
]

# Demographic categorical features
demographic_cat = [
    "sex",
    "language",
    "insurance_type",
    "age_group"
]

# Visit context categorical features
context_cat = [
    "arrival_mode",
    "transport_origin",
    "pain_location",
    "mental_status_triage",
    "chief_complaint_system"
]

# Count / utilization features
count_cols = [
    "num_prior_ed_visits_12m",
    "num_prior_admissions_12m",
    "num_active_medications",
    "num_comorbidities"
]

# Continuous vital signs
vital_continuous = [
    "systolic_bp",
    "diastolic_bp",
    "mean_arterial_pressure",
    "pulse_pressure",
    "heart_rate",
    "respiratory_rate",
    "temperature_c",
    "spo2"
]

# Clinical scores (ordinal / discrete)
clinical_scores = [
    "gcs_total",
    "pain_score",
    "news2_score"
]

# Anthropometric continuous features
body_measurements = [
    "weight_kg",
    "height_cm",
    "bmi"
]

# Derived physiological metrics
derived_features = [
    "shock_index"
]

# Outcome / target variables
target_cols = [
    "triage_acuity"
]

# Post-visit outcomes (should not be used as predictors if predicting triage)
post_visit_outcomes = [
    "disposition",
    "ed_los_hours"
]

**predicting triage_acuity, then do NOT use**<br>
**disposition**<br>
**ed_los_hours**<br>
because they happen after triage → this would cause data leakage.

# `Demographic Categorical Features.`
These are non-numeric variables describing patient characteristics.


In [ ]:
for i in demographic_cat:
    print(f"Value counts for {i}:")
    print(df_train[i].value_counts())
    print("\n")

## Demographic and Socioeconomic Feature Distributions

This section examines the **value counts of key demographic and socioeconomic variables**, including `sex`, `language`, `insurance_type`, and `age_group`.  
Understanding these distributions helps identify **population characteristics and potential biases in the dataset**.

---

### Sex Distribution

| Sex | Count |
|-----|------|
| F | 40,339 |
| M | 37,735 |
| Other | 1,926 |

**Observations**

- The dataset shows a **slightly higher number of female patients** compared to males.
- A small portion of patients are categorized as **Other**, indicating non-binary or unspecified gender identities.
- The overall distribution appears **reasonably balanced**, which is beneficial for modeling fairness.

---

### Language Distribution

| Language | Count |
|----------|------|
| Finnish | 44,134 |
| English | 8,024 |
| Swedish | 6,315 |
| Russian | 5,587 |
| Estonian | 4,858 |
| Other | 3,968 |
| Arabic | 3,944 |
| Somali | 3,170 |

**Observations**

- **Finnish dominates the dataset**, which aligns with the dataset likely originating from Finland.
- Several minority languages are represented, including **English, Swedish, Russian, Estonian, Arabic, and Somali**.
- Language diversity may influence **communication barriers and triage assessment**, which could indirectly impact clinical outcomes.

---

### Insurance Type Distribution

| Insurance Type | Count |
|----------------|------|
| Public | 48,170 |
| Private | 19,915 |
| None | 6,320 |
| Military | 3,196 |
| Unknown | 2,399 |

**Observations**

- The majority of patients use **public insurance**, which is expected in healthcare systems with strong public coverage.
- A substantial portion of patients have **private insurance**, indicating mixed healthcare access.
- Some patients have **no insurance or unknown status**, which may correlate with socioeconomic factors.

---

### Age Group Distribution

| Age Group | Count |
|-----------|------|
| Middle-aged | 27,889 |
| Young adult | 23,863 |
| Elderly | 21,653 |
| Pediatric | 6,595 |

**Observations**

- The largest group is **middle-aged adults**, followed by **young adults and elderly patients**.
- **Pediatric patients represent the smallest proportion**, which is typical for general emergency department datasets.
- Age distribution is important because **triage severity often varies significantly across age groups**.

---

### Summary

The dataset contains **diverse demographic characteristics** across sex, language, insurance coverage, and age groups.  
These variables may influence **healthcare access, communication, and clinical risk profiles**, making them important contextual features for triage prediction models.

# `Context Categorical Features`

In [ ]:
for i in context_cat:
    print(f"Value counts for {i}:")
    print(df_train[i].value_counts())
    print("\n")

## Emergency Visit Context Feature Distributions

This section explores the distribution of **visit context variables**, which describe **how patients arrive at the emergency department, where they come from, and the primary clinical presentation at triage**.

These variables provide important contextual information that may influence **triage decisions and severity assessment**.

---

## Arrival Mode

| Arrival Mode | Count |
|---|---|
| walk-in | 38,459 |
| ambulance | 22,404 |
| transfer | 7,987 |
| brought_by_family | 5,583 |
| police | 3,162 |
| helicopter | 2,405 |

### Observations

- **Walk-in patients represent the largest group**, indicating that many emergency visits are self-initiated.
- **Ambulance arrivals are the second largest group**, often associated with **higher acuity cases**.
- **Helicopter and police transports are rare**, typically linked to **severe trauma or critical emergencies**.
- Arrival mode can serve as an **important proxy for initial clinical severity**.

---

## Transport Origin

| Origin | Count |
|---|---|
| school | 11,571 |
| workplace | 11,499 |
| outdoor | 11,459 |
| public_space | 11,430 |
| nursing_home | 11,393 |
| other_hospital | 11,339 |
| home | 11,309 |

### Observations

- The distribution is **very balanced across transport origins**.
- Patients arrive from a wide variety of environments including **schools, workplaces, public spaces, and nursing homes**.
- **Nursing home origins may be associated with elderly patients and higher comorbidity burdens**.
- **Other hospital transfers** may indicate **patients requiring specialized care or advanced treatment**.

---

## Pain Location

| Pain Location | Count |
|---|---|
| pelvis | 8,998 |
| back | 8,982 |
| extremity | 8,920 |
| abdomen | 8,907 |
| chest | 8,899 |
| head | 8,871 |
| unknown | 8,844 |
| none | 8,794 |
| multiple | 8,785 |

### Observations

- Pain locations are **distributed fairly evenly across body regions**.
- **Chest pain is clinically significant** as it may indicate **cardiac emergencies**.
- **Head pain** could be related to neurological conditions such as migraines, trauma, or stroke.
- **Abdominal pain** is one of the most common emergency complaints and may represent diverse conditions.

---

## Mental Status at Triage

| Mental Status | Count |
|---|---|
| alert | 46,212 |
| confused | 14,289 |
| drowsy | 9,101 |
| agitated | 5,915 |
| unresponsive | 4,483 |

### Observations

- The majority of patients arrive **alert**, indicating normal consciousness.
- **Confusion and drowsiness represent significant portions**, which may indicate neurological, metabolic, or infectious problems.
- **Unresponsive patients are fewer but represent critical emergencies**.
- Mental status is a **key indicator used in triage scoring systems such as GCS and NEWS2**.

---

## Chief Complaint System

| System | Count |
|---|---|
| gastrointestinal | 5,812 |
| respiratory | 5,804 |
| infectious | 5,764 |
| endocrine | 5,749 |
| musculoskeletal | 5,733 |
| trauma | 5,726 |
| genitourinary | 5,724 |
| cardiovascular | 5,718 |
| neurological | 5,691 |
| psychiatric | 5,688 |
| ENT | 5,680 |
| ophthalmic | 5,644 |
| other | 5,642 |
| dermatological | 5,625 |

### Observations

- The distribution across complaint systems is **remarkably balanced**, suggesting broad representation of clinical conditions.
- **Cardiovascular and neurological complaints are particularly important**, as they often correspond to high-acuity emergencies.
- **Respiratory and infectious complaints are also common**, especially in seasonal disease patterns.
- Chief complaint categories help clinicians **quickly route patients to appropriate diagnostic pathways**.

---

## Summary

These visit context variables describe **how patients enter the emergency care system and the nature of their presenting symptoms**.  
They provide valuable signals for predicting **triage acuity and clinical severity**, as factors such as **arrival mode, mental status, and chief complaint system are strongly linked to emergency risk levels**.<br>
`Hindi`<br>
Dataset balanced and realistic ED population jaisa lag raha hai

# `Count Features`

In [ ]:
for i in count_cols:
    print(f"Summary value_counts for {i}:")
    print(df_train[i].value_counts())
    print("\n")

## Patient History and Healthcare Utilization Features

This section explores **count-based clinical history variables** that capture a patient's **prior healthcare utilization, medication burden, and comorbidity load**.  
These features provide insight into a patient's **baseline health complexity and healthcare dependency**, which can strongly influence emergency triage severity.

---

## Prior Emergency Department Visits (Last 12 Months)

| Visits | Count |
|---|---|
| 0 | 26,471 |
| 1 | 23,495 |
| 2 | 15,042 |
| 3 | 8,077 |
| 4 | 3,956 |
| 5 | 1,752 |
| 6 | 785 |
| 7 | 276 |
| 8 | 110 |
| 9 | 27 |
| 10+ | very rare |

### Observations

- A large proportion of patients have **0–2 prior ED visits**, indicating occasional emergency care usage.
- The distribution is **right-skewed**, meaning a small group of patients are **frequent ED users**.
- High prior ED visits may indicate:
  - chronic illness
  - poor outpatient care access
  - complex medical conditions.

---

## Prior Hospital Admissions (Last 12 Months)

| Admissions | Count |
|---|---|
| 0 | 57,067 |
| 1 | 15,568 |
| 2 | 5,282 |
| 3 | 1,510 |
| 4+ | very rare |

### Observations

- The majority of patients **had no hospital admissions in the past year**.
- A smaller subset experienced **repeated admissions**, suggesting higher disease severity or chronic conditions.
- Frequent hospitalizations may signal **patients at higher clinical risk during ED visits**.

---

## Number of Active Medications

| Medications | Count |
|---|---|
| 0–2 | common |
| 3–6 | moderate |
| 7–10 | frequent |
| 10+ | smaller group |

### Observations

- The distribution shows **polypharmacy patterns**, where many patients take multiple medications.
- Patients taking **many medications often have multiple chronic diseases**.
- High medication counts may increase risk of:
  - drug interactions
  - adverse drug reactions
  - complex clinical presentations.

---

## Number of Comorbidities

| Comorbidities | Count |
|---|---|
| 2–6 | most common |
| 7–10 | moderate |
| 10+ | smaller subset |

### Observations

- Most patients have **multiple underlying medical conditions**.
- The distribution suggests a **high comorbidity burden in the population**.
- Higher comorbidity counts are associated with:
  - increased clinical complexity
  - greater risk of severe outcomes
  - higher healthcare utilization.

---

## Summary

These utilization and health complexity features capture important aspects of a patient's **medical background and healthcare usage patterns**.

Key insights include:

- Many patients have **previous ED visits**, indicating repeated emergency care utilization.
- Most patients **have not been hospitalized recently**, but a small subset shows frequent admissions.
- Medication counts suggest **moderate to high levels of polypharmacy**.
- Many patients have **multiple comorbidities**, reflecting complex health profiles.

Together, these variables provide strong signals about **patient health complexity and risk level**, which may significantly influence **triage acuity prediction**.




Most patients have **low prior healthcare utilization** (few ED visits and admissions), but many show **multiple comorbidities and ongoing medications**, suggesting a population with **moderate chronic disease burden** that may influence emergency triage severity.